<a href="https://colab.research.google.com/github/Otza02/land2vec/blob/main/notebooks/prueba_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/Otza02/land2vec.git
%cd /content/land2vec
!pip install -e .

Cloning into 'land2vec'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 93 (delta 35), reused 76 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 4.01 MiB | 10.25 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/land2vec


### ---Restart session before next cell---

In [ ]:
%cd /content/land2vec

In [ ]:
import tqdm
from datetime import datetime

import torch
from torch.utils.data import DataLoader, random_split
from torch.optim import lr_scheduler

from land2vec.dataset import load_data, SequenceDataset
from land2vec.config import Config
from land2vec.model import DecoderTransformer, run_epoch
from land2vec.tokenizer import Tokenizer
from land2vec.model_explicit import GPTDecoder

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
config = Config(patience=4)

torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
device

In [ ]:
print("loading data")
file_path = "data/id_seqs_text_2000_2022_chaco_santiago_frontier.zip"
try:
    dataset = load_data(file_path="/content/land2vec/" + file_path, window=config.block_size)
except FileNotFoundError:
    dataset = load_data(file_path="../data/seqs_short.csv", window=config.block_size)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

generator = torch.Generator().manual_seed(config.seed)

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], generator=generator)
loader_kwargs = dict(
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    pin_memory=True,
    persistent_workers=True,
)

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **loader_kwargs,
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **loader_kwargs,
)

loading data


100%|██████████| 1424457/1424457 [00:52<00:00, 26979.53it/s]


In [4]:
config

Config(block_size=8, n_embd=32, n_head=2, n_layer=2, dropout=0.1, epochs=25, patience=5, batch_size=128, lr=0.001, min_lr=1e-06, weight_decay=0.01)

In [ ]:
model = GPTDecoder(
    vocab_size=len(Tokenizer.VOCAB),
    block_size=config.block_size,
    n_embd=config.n_embd,
    n_head=config.n_head,
    n_layer=config.n_layer,
    dropout=config.dropout,
).to(device)
model = torch.compile(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay, fused=device == "cuda")
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs, eta_min=config.min_lr)
scaler = torch.amp.GradScaler(enabled=True)

history = {
    "train_loss": [],
    "val_loss": [],
    "lr": [],
    "epoch_time": [],
    "tokens_per_sec": [],
}

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0

val_eval_epoch = 20

total_params = sum(p.numel() for p in model.parameters())

print(f"parameters: {total_params:,}")

In [ ]:
global_start = datetime.now()
for epoch in range(config.epochs):
    start = datetime.now()
    train_loss = run_epoch(model, train_loader, optimizer, device, scaler=scaler, use_amp=True)
    val_loss = None
    
    if epoch == 0 or (epoch + 1) % config.validate_every == 0:
        val_loss = run_epoch(model, val_loader, optimizer=None, device=device, scaler=None, use_amp=True)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": config,
                    "epoch": epoch,
                    "val_loss": val_loss,
                },
                "best_model.pt",
            )
            patience_counter = 0
        else:
            patience_counter += 1

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    epoch_seconds = (datetime.now() - start).total_seconds()

    tokens_processed = (len(train_loader.dataset) * config.block_size)
    tokens_per_sec = (tokens_processed / epoch_seconds)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(current_lr)
    history["epoch_time"].append(epoch_seconds)
    history["tokens_per_sec"].append(tokens_per_sec)

    ppl = torch.math.exp(train_loss)
    print(
        f"[Epoch {epoch+1:03d}] "
        f"train={train_loss:.4f} "
        f"val={val_loss if val_loss is not None else 'skip'} "
        f"ppl={ppl:.2f} "
        f"lr={current_lr:.2e} "
        f"time={epoch_seconds:.1f}s "
        f"token/s={tokens_per_sec:,.0f}"
    )

    if patience_counter >= config.patience:
        print("Early stopping triggered")
        break

total_time = (datetime.now() - global_start).seconds

print(f"best epoch: {best_epoch}")
print(f"best val loss: {best_val_loss:.4f}")
print(f"total training time: {total_time//60:02}:{total_time%60:02}")

# load best
checkpoint = torch.load("best_model.pt", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])

In [ ]:
torch.save(model.state_dict(), "first-test.pt")

In [ ]:
model = DecoderTransformer(
    vocab_size=len(Tokenizer.VOCAB),
    block_size=config.block_size,
    n_embd=config.n_embd,
    n_head=config.n_head,
    n_layer=config.n_layer,
).to(device)

model.load_state_dict(torch.load("models/first-test.pt"))

model.eval()

GPT(
  (token_embedding): Embedding(16, 32)
  (position_embedding): Embedding(8, 32)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=32, out_features=16, bias=True)
)

In [ ]:
model.eval()

correct = 0
total = 0

all_preds = []
all_targets = []

with torch.no_grad():
    for x, y in tqdm.tqdm(test_loader):
        x = x.long().to(device)
        y = y.long().to(device)

        logits, loss = model(x, y)

        preds = torch.argmax(logits, dim=-1)

        correct += (preds == y).sum().item()
        total += y.numel()

        all_preds.append(preds.cpu())
        all_targets.append(y.cpu())

accuracy = correct / total

print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.9953
